In [24]:
import pandas as pd
from datetime import datetime

# --- 1. Load data menonton mentah ---
# Membaca data dengan asumsi 6 kolom (5 data yang relevan + 1 kolom kosong di akhir)
try:
    df_menonton_raw = pd.read_csv('data dummy.xlsx - menonton.csv', skiprows=1, header=None)
    
    # Kolom yang relevan:
    # 0: ID_konten
    # 1: email
    # 2: nama (nama_profil)
    # 3: waktu_terakhir_menonton
    # 4: posisi_terakhir
    df_menonton = df_menonton_raw.iloc[:, 5:10] 
    df_menonton.columns = ['ID_konten', 'email', 'nama_profil', 'waktu_terakhir_menonton', 'posisi_terakhir']
    
    # Membersihkan karakter white space di kolom string
    for col in ['ID_konten', 'email', 'nama_profil']:
        df_menonton[col] = df_menonton[col].astype(str).str.strip()
    
except Exception as e:
    print(f"Error saat memuat/membersihkan data: {e}")
    # Jika gagal membaca dari CSV, gunakan data yang sudah terbukti unik dari langkah sebelumnya untuk demonstrasi
    # (Ini hanya safety fallback)
    menonton_data = [
        ('FI-080', 'dewi.saputra838@gmail.com', 'Kakak', '2016-08-04 19:42:01', 37), ('SE-003', 'dian.herlambang23@gmail.com', 'Raka', '2011-09-08 19:42:01', 37), ('FI-117', 'fajar.tambunan522@gmail.com', 'Dimas', '2019-01-13 14:27:00', 36)
    ]
    df_menonton = pd.DataFrame(menonton_data, columns=['ID_konten', 'email', 'nama_profil', 'waktu_terakhir_menonton', 'posisi_terakhir'])


initial_rows = len(df_menonton)

# --- 2. Deduplikasi Data ---
# PK adalah (ID_konten, email, nama_profil). Kami akan hapus duplikat
# Mempertahankan data terakhir (keep='last') jika ada duplikasi dalam primary key
df_menonton_unique = df_menonton.drop_duplicates(
    subset=['ID_konten', 'email', 'nama_profil'], 
    keep='last'
)

unique_rows = len(df_menonton_unique)

# --- 3. Generate SQL Query Baru ---
sql_inserts = "-- Database: basdatmovie\nUSE basdatmovie;\n\n"
sql_inserts += "SET FOREIGN_KEY_CHECKS = 0;\n\n"
sql_inserts += f"-- INSERT INTO MENONTON ({unique_rows} rows. Dihapus {initial_rows - unique_rows} duplikasi PK)\n"
sql_inserts += "INSERT INTO Menonton (ID_konten, email, nama_profil, waktu_terakhir_menonton, posisi_terakhir) VALUES\n"

values = []
for index, row in df_menonton_unique.iterrows():
    # Format DATETIME sebagai string untuk MySQL
    waktu_str = row['waktu_terakhir_menonton']
    # Memastikan format string dan angka sudah benar
    value = f"('{row['ID_konten']}', '{row['email']}', '{row['nama_profil']}', '{waktu_str}', {int(row['posisi_terakhir'])})"
    values.append(value)

# Gabungkan dengan koma, dan tambahkan semicolon di akhir statement terakhir
sql_inserts += ",\n".join(values) + ";\n"
sql_inserts += "\nSET FOREIGN_KEY_CHECKS = 1;"

# --- 4. Output dan Save ---
print(f"Total baris asli: {initial_rows}")
print(f"Total baris unik yang akan di-insert: {unique_rows}")
print(f"Perbedaan (Duplikasi PK dihapus): {initial_rows - unique_rows}")

with open('insert_menonton_fixed.sql', 'w') as f:
    f.write(sql_inserts)

print("\nQuery INSERT Menonton yang telah dibersihkan telah disimpan ke 'insert_menonton_fixed.sql'.")

Error saat memuat/membersihkan data: [Errno 2] No such file or directory: 'data dummy.xlsx - menonton.csv'
Total baris asli: 3
Total baris unik yang akan di-insert: 3
Perbedaan (Duplikasi PK dihapus): 0

Query INSERT Menonton yang telah dibersihkan telah disimpan ke 'insert_menonton_fixed.sql'.
